In [ ]:
import cv2
import base64
import json
import numpy as np
from IPython.display import HTML, display

def select_roi(image):
    """Muestra en pantalla la seleccion en formato (x0, y0) a (x1, y1)"""

    # 1. Preparar la imagen
    _, buffer = cv2.imencode('.jpg', image)
    img_base64 = base64.b64encode(buffer).decode('utf-8')

    # 2. El HTML/JS (Ajustado para Jupyter estándar)
    # Usamos un input oculto para "almacenar" el resultado que Python pueda leer
    div_id = "roi_container"

    canvas_html = f"""
    <div id="{div_id}" style="position: relative; display: inline-block; cursor: crosshair;">
        <canvas id="c_img"></canvas>
        <canvas id="c_draw" style="position: absolute; left: 0; top: 0;"></canvas>
    </div>
    <div style="margin-top: 10px;">
        <span id="output_text" style="margin-left: 15px; font-family: sans-serif;">Arrastra el mouse...</span>
    </div>

    <script>
        (function() {{
            const canvasImg = document.getElementById('c_img');
            const canvasDraw = document.getElementById('c_draw');
            const ctxImg = canvasImg.getContext('2d');
            const ctxDraw = canvasDraw.getContext('2d');
            const outText = document.getElementById('output_text');

            let img = new Image();
            img.src = "data:image/jpeg;base64,{img_base64}";
            img.onload = () => {{
                canvasImg.width = canvasDraw.width = img.width;
                canvasImg.height = canvasDraw.height = img.height;
                ctxImg.drawImage(img, 0, 0);
            }};

            let x0, y0, x1, y1, drawing = false;

            canvasDraw.onmousedown = (e) => {{
                const r = canvasDraw.getBoundingClientRect();
                x0 = e.clientX - r.left; y0 = e.clientY - r.top;
                drawing = true;
            }};

            canvasDraw.onmousemove = (e) => {{
                if (!drawing) return;
                const r = canvasDraw.getBoundingClientRect();
                x1 = e.clientX - r.left; y1 = e.clientY - r.top;
                ctxDraw.clearRect(0, 0, canvasDraw.width, canvasDraw.height);
                ctxDraw.strokeStyle = "#00FF00";
                ctxDraw.lineWidth = 2;
                ctxDraw.strokeRect(x0, y0, x1 - x0, y1 - y0);
                outText.innerText = `ROI: (${{Math.round(x0)}}, ${{Math.round(y0)}}) a (${{Math.round(x1)}}, ${{Math.round(y1)}})`;
            }};

            canvasDraw.onmouseup = () => drawing = false;
        }})();
    </script>
    """
    display(HTML(canvas_html))

# --- Uso ---
import cv2
img = cv2.imread('grid_symbols.png')

select_roi(img)